# Lab03 — Monte Carlo Methods trong Học tăng cường

**Học phần:** Học tăng cường – Reinforcement Learning  
**Số bài:** 36 · **Ngôn ngữ:** Python (Google Colab / Jupyter)

In [ ]:
STUDENT_ID = "MSSV"
STUDENT_NAME = "Ho Ten"

MAIN_ENV = "Blackjack-v1"
OPTIONAL_ENVS = [
    "FrozenLake-v1",
]
print(STUDENT_ID, STUDENT_NAME, MAIN_ENV, OPTIONAL_ENVS)

## 0. Cài đặt & thư viện

In [ ]:
!pip install -q "gymnasium[toy-text,classic-control]"

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

print("Gymnasium:", gym.__version__)
print("NumPy:", np.__version__)

### Thư viện hàm tự cài đặt
Các hàm dưới đây giống file `src/mc_utils.py` và `src/experiments.py` (đặt trong notebook để chạy độc lập trên Colab). **Không dùng thư viện RL có sẵn.**

In [ ]:
"""
mc_utils.py - Cac ham dung chung cho Lab03: Monte Carlo Methods.

Tat ca thuat toan (prediction, control, epsilon-greedy) deu tu cai dat,
khong dung thu vien RL co san. Chi dung gymnasium de lay environment.
"""
import os
import pickle
from collections import defaultdict

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- hang so
ACTION_NAMES = {0: "stick", 1: "hit"}   # 0 = dung lai, 1 = rut them bai
STICK, HIT = 0, 1

try:                                   # chay tu file .py
    _BASE = os.path.dirname(os.path.abspath(__file__))
    FIG_DIR = os.path.join(_BASE, "..", "figures")
    DATA_DIR = os.path.join(_BASE, "..", "data")
except NameError:                      # chay trong notebook
    FIG_DIR, DATA_DIR = "figures", "data"


def make_env(name="Blackjack-v1", **kwargs):
    """Tao environment gymnasium theo ten."""
    return gym.make(name, **kwargs)


# ---------------------------------------------------------------- policy
def make_random_policy(n_actions, rng):
    """Tra ve policy ngau nhien deu: state -> action."""
    return lambda state: int(rng.integers(n_actions))


def stick_on_20_policy(state):
    """Policy co dinh cua Bai 11: stick (0) neu tong >= 20, nguoc lai hit (1)."""
    player_sum, dealer_card, usable_ace = state
    if player_sum >= 20:
        return STICK
    return HIT


def stick_on_17_policy(state):
    """Policy co dinh bo sung (giong dealer): stick neu tong >= 17."""
    return STICK if state[0] >= 17 else HIT


def make_policy_fn(policy_dict, default_action=STICK):
    """Doi dict {state: action} thanh ham policy (state la khoa thieu -> default_action)."""
    return lambda state: policy_dict.get(state, default_action)


# ---------------------------------------------------------------- episode
def generate_episode(env, policy, seed=None, max_steps=1000):
    """
    Sinh MOT episode bang cach cho agent tuong tac voi env theo `policy`.

    Tham so
    -------
    env : environment gymnasium
    policy : ham state -> action
    seed : seed cho env.reset (None = tiep tuc luong ngau nhien cua env)
    max_steps : gioi han so buoc (phong env khong ket thuc)

    Tra ve
    ------
    episode : list cac tuple (state, action, reward);
              reward la R_(t+1) nhan duoc sau khi lam action o state S_t.
    """
    state, _ = env.reset(seed=seed)
    episode = []
    for _ in range(max_steps):
        action = policy(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if terminated or truncated:
            break
    return episode


def compute_returns(rewards, gamma=1.0):
    """
    Tinh return G_t cho moi timestep, duyet NGUOC tu cuoi episode:
        G_t = R_(t+1) + gamma * G_(t+1)

    Tra ve list [G_0, G_1, ..., G_(T-1)] cung do dai voi `rewards`.
    """
    returns = [0.0] * len(rewards)
    g = 0.0
    for t in reversed(range(len(rewards))):
        g = rewards[t] + gamma * g
        returns[t] = g
    return returns


# ---------------------------------------------------------------- prediction
def _mc_prediction(env, policy, n_episodes, gamma, first_visit, seed, history):
    """Ham loi dung chung cho First-Visit va Every-Visit MC prediction."""
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)
    for i in range(n_episodes):
        episode = generate_episode(env, policy, seed=seed if i == 0 else None)
        states = [s for s, _, _ in episode]
        G = compute_returns([r for _, _, r in episode], gamma)
        seen = set()
        for t, state in enumerate(states):
            if first_visit:
                if state in seen:          # chi lay lan xuat hien dau tien
                    continue
                seen.add(state)
            returns_sum[state] += G[t]
            returns_count[state] += 1
        if history is not None and (i + 1) in history["checkpoints"]:
            for s in history["track"]:
                c = returns_count.get(s, 0)
                history.setdefault(s, []).append(returns_sum[s] / c if c else np.nan)
    V = {s: returns_sum[s] / returns_count[s] for s in returns_count}
    return V, dict(returns_count)


def first_visit_mc_prediction(env, policy, n_episodes, gamma=1.0, seed=None, history=None):
    """
    First-Visit MC prediction: V(s) = trung binh return tai LAN DAU s xuat hien trong episode.

    history (tuy chon): dict {"checkpoints": [...], "track": [states]} se duoc dien them
    estimate V(s) tai cac checkpoint de ve duong hoi tu.
    Tra ve (V, returns_count).
    """
    return _mc_prediction(env, policy, n_episodes, gamma, True, seed, history)


def every_visit_mc_prediction(env, policy, n_episodes, gamma=1.0, seed=None, history=None):
    """Every-Visit MC prediction: dung return o MOI lan s xuat hien. Tra ve (V, returns_count)."""
    return _mc_prediction(env, policy, n_episodes, gamma, False, seed, history)


def mc_action_value_prediction(env, policy, n_episodes, gamma=1.0, seed=None):
    """
    Uoc luong Q(s,a) cua `policy` bang first-visit MC tren cap (s,a).
    Tra ve (Q, N): Q[state][action] va N[state][action] (so lan cap duoc cap nhat).
    """
    n_actions = env.action_space.n
    returns_sum = defaultdict(lambda: np.zeros(n_actions))
    N = defaultdict(lambda: np.zeros(n_actions))
    for i in range(n_episodes):
        episode = generate_episode(env, policy, seed=seed if i == 0 else None)
        G = compute_returns([r for _, _, r in episode], gamma)
        seen = set()
        for t, (s, a, _) in enumerate(episode):
            if (s, a) in seen:
                continue
            seen.add((s, a))
            returns_sum[s][a] += G[t]
            N[s][a] += 1
    Q = defaultdict(lambda: np.zeros(n_actions))
    for s in N:
        Q[s] = np.divide(returns_sum[s], N[s], out=np.zeros(n_actions), where=N[s] > 0)
    return Q, N


# ---------------------------------------------------------------- epsilon-greedy
def greedy_action(Q, state, n_actions):
    """Action tot nhat theo Q (np.argmax). State chua gap -> action 0."""
    if state not in Q:
        return 0
    return int(np.argmax(Q[state][:n_actions]))


def epsilon_greedy_action(Q, state, n_actions, epsilon, rng):
    """
    Epsilon-greedy: voi xac suat epsilon chon ngau nhien (deu) trong moi action,
    nguoc lai chon greedy. => P(greedy) = 1 - epsilon + epsilon/n_actions.
    """
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))
    return greedy_action(Q, state, n_actions)


# ---------------------------------------------------------------- control
def update_q_incremental(Q, N, state, action, G):
    """
    Cap nhat trung binh tang dan (khong can luu moi return):
        N(s,a) += 1
        Q(s,a) += (G - Q(s,a)) / N(s,a)
    """
    N[state][action] += 1
    Q[state][action] += (G - Q[state][action]) / N[state][action]


def on_policy_mc_control(env, n_episodes, gamma=1.0, epsilon=0.1, seed=42):
    """
    On-policy First-Visit MC control voi epsilon-greedy.

    Moi episode: sinh bang epsilon-greedy(Q hien tai) -> tinh return ->
    first-visit update Q(s,a) -> policy (epsilon-greedy theo Q) tu dong cai thien.

    Tra ve
    ------
    Q : defaultdict state -> mang Q(s, .)
    policy : dict {state: greedy action} cho cac state da gap
    episode_rewards : list tong reward cua tung episode luc train
    """
    n_actions = env.action_space.n
    rng = np.random.default_rng(seed)
    Q = defaultdict(lambda: np.zeros(n_actions))
    N = defaultdict(lambda: np.zeros(n_actions))
    behavior = lambda s: epsilon_greedy_action(Q, s, n_actions, epsilon, rng)
    episode_rewards = []
    for i in range(n_episodes):
        episode = generate_episode(env, behavior, seed=seed if i == 0 else None)
        rewards = [r for _, _, r in episode]
        G = compute_returns(rewards, gamma)
        episode_rewards.append(sum(rewards))
        seen = set()
        for t, (s, a, _) in enumerate(episode):
            if (s, a) in seen:
                continue
            seen.add((s, a))
            update_q_incremental(Q, N, s, a, G[t])
    policy = {s: int(np.argmax(q)) for s, q in Q.items()}
    return Q, policy, episode_rewards


def load_or_train_control(env, n_episodes=300_000, gamma=1.0, epsilon=0.1, seed=42):
    """Train MC control, luu ket qua vao data/ de cac bai sau dung lai (tiet kiem thoi gian)."""
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, f"mc_control_{n_episodes}_{gamma}_{epsilon}_{seed}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            Q, policy, rewards = pickle.load(f)
        Q = defaultdict(lambda: np.zeros(env.action_space.n), Q)
        return Q, policy, rewards
    Q, policy, rewards = on_policy_mc_control(env, n_episodes, gamma, epsilon, seed)
    with open(path, "wb") as f:
        pickle.dump((dict(Q), policy, rewards), f)
    return Q, policy, rewards


# ---------------------------------------------------------------- evaluation
def evaluate_policy(env, policy, n_episodes=10000, seed=123):
    """
    Danh gia policy (khong hoc, khong explore) tren episode MOI.
    Tra ve dict: win_rate, loss_rate, draw_rate, mean_reward, std_error.
    """
    totals = []
    for i in range(n_episodes):
        ep = generate_episode(env, policy, seed=seed if i == 0 else None)
        totals.append(sum(r for _, _, r in ep))
    totals = np.array(totals)
    return {
        "win_rate": float(np.mean(totals > 0)),
        "loss_rate": float(np.mean(totals < 0)),
        "draw_rate": float(np.mean(totals == 0)),
        "mean_reward": float(totals.mean()),
        "std_error": float(totals.std() / np.sqrt(n_episodes)),
    }


# ---------------------------------------------------------------- ve hinh
def moving_average(x, window=1000):
    """Trung binh truot (kieu 'valid')."""
    x = np.asarray(x, dtype=float)
    return np.convolve(x, np.ones(window) / window, mode="valid")


def save_fig(fig, name):
    """Luu hinh vao figures/ roi hien thi."""
    os.makedirs(FIG_DIR, exist_ok=True)
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Da luu:", os.path.normpath(path))


def value_grid(V, usable_ace, sums=range(12, 22), dealers=range(1, 11)):
    """Bang V(player_sum, dealer_card) cho Blackjack; o chua gap = NaN."""
    return np.array([[V.get((p, d, usable_ace), np.nan) for d in dealers] for p in sums])


def plot_value_heatmap(V, usable_ace, name, title=None):
    """Heatmap V(s) theo player sum (12-21) x dealer showing (A,2..10)."""
    sums, dealers = list(range(12, 22)), list(range(1, 11))
    grid = value_grid(V, usable_ace, sums, dealers)
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(grid, origin="lower", cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(10)); ax.set_xticklabels(["A"] + [str(d) for d in dealers[1:]])
    ax.set_yticks(range(10)); ax.set_yticklabels(sums)
    ax.set_xlabel("Dealer showing"); ax.set_ylabel("Player sum")
    ax.set_title(title or f"V(s), usable_ace={usable_ace}")
    fig.colorbar(im, label="V(s)")
    save_fig(fig, name)


def plot_policy_heatmap(policy, usable_ace, ax, title):
    """Ve policy hoc duoc (0=stick, 1=hit) tren mot truc ax."""
    sums, dealers = list(range(12, 22)), list(range(1, 11))
    grid = np.array([[policy.get((p, d, usable_ace), np.nan) for d in dealers] for p in sums])
    ax.imshow(grid, origin="lower", cmap="coolwarm", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(10)); ax.set_xticklabels(["A"] + [str(d) for d in dealers[1:]])
    ax.set_yticks(range(10)); ax.set_yticklabels(sums)
    ax.set_xlabel("Dealer showing"); ax.set_ylabel("Player sum"); ax.set_title(title)
    for i in range(10):
        for j in range(10):
            v = grid[i, j]
            if not np.isnan(v):
                ax.text(j, i, "H" if v == 1 else "S", ha="center", va="center", fontsize=8)

In [ ]:
"""
experiments.py - Cac thi nghiem / ham ve hinh cap cao cho Lab03.
Dung cac ham da tu cai dat trong mc_utils.py.
"""
import numpy as np
import matplotlib.pyplot as plt



def sample_long_episode(env, policy, min_len=3, max_tries=1000, seed0=0):
    """Tim mot episode co it nhat `min_len` buoc de minh hoa trajectory (thu lan luot cac seed)."""
    episode = []
    for k in range(max_tries):
        episode = generate_episode(env, policy, seed=seed0 + k)
        if len(episode) >= min_len:
            return episode
    return episode


def compare_first_every_convergence(blackjack_env, frozen_env, n_episodes=10000, seed=0):
    """
    Bai 23: ve V(s) cua First-Visit va Every-Visit theo so episode cho 3 state,
    tren Blackjack (hang tren) va FrozenLake (hang duoi). Luu first_vs_every_visit.png.
    Tra ve dict mean absolute difference cua tung env.
    """
    checkpoints = [c for c in (100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000) if c <= n_episodes]
    configs = [
        ("Blackjack-v1, stick_on_20", blackjack_env, lambda: stick_on_20_policy,
         [(20, 10, False), (18, 6, False), (13, 2, False)]),
        ("FrozenLake-v1, random policy", frozen_env,
         lambda: make_random_policy(frozen_env.action_space.n, np.random.default_rng(seed)),
         [0, 4, 9]),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
    diffs = {}
    for row, (title, env, policy_factory, states) in enumerate(configs):
        hist_f = {"checkpoints": set(checkpoints), "track": states}
        hist_e = {"checkpoints": set(checkpoints), "track": states}
        # policy_factory() tao policy moi (rng moi) => hai phuong phap thay CUNG chuoi episode
        V_f, _ = first_visit_mc_prediction(env, policy_factory(), n_episodes, seed=seed, history=hist_f)
        V_e, _ = every_visit_mc_prediction(env, policy_factory(), n_episodes, seed=seed, history=hist_e)
        common = set(V_f) & set(V_e)
        diffs[title] = float(np.mean([abs(V_f[s] - V_e[s]) for s in common]))
        for col, s in enumerate(states):
            ax = axes[row, col]
            ax.plot(checkpoints, hist_f[s], "o-", label="First-Visit")
            ax.plot(checkpoints, hist_e[s], "s--", label="Every-Visit")
            ax.set_xscale("log"); ax.set_title(f"{title}\nstate = {s}", fontsize=9)
            ax.set_xlabel("số episode"); ax.set_ylabel("V(s)"); ax.grid(alpha=0.3)
            if row == 0 and col == 0:
                ax.legend()
    fig.suptitle("First-Visit vs Every-Visit MC: hội tụ của V(s)")
    fig.tight_layout()
    save_fig(fig, "first_vs_every_visit.png")
    return diffs


def epsilon_experiment(env, epsilons=(0.01, 0.05, 0.10, 0.20, 0.50), n_samples=10000,
                       n_train=100_000, n_eval=20_000, seed=0):
    """
    Bai 29: (a) tan suat chon greedy action cua epsilon-greedy voi Q gia [1, 5];
            (b) chat luong policy hoc duoc (mean reward khi danh gia) theo epsilon luc train.
    Luu epsilon_comparison.png. Tra ve list cac dong ket qua.
    """
    rng = np.random.default_rng(seed)
    Q_toy = {"s": np.array([1.0, 5.0])}       # action 1 la greedy
    rows = []
    for eps in epsilons:
        actions = np.array([epsilon_greedy_action(Q_toy, "s", 2, eps, rng) for _ in range(n_samples)])
        freq = float(np.mean(actions == 1))
        _, policy, _ = on_policy_mc_control(env, n_train, epsilon=eps, seed=seed)
        ev = evaluate_policy(env, make_policy_fn(policy), n_eval, seed=123)
        rows.append({"epsilon": eps, "greedy_freq": freq, "theory": 1 - eps + eps / 2,
                     "mean_reward": ev["mean_reward"], "std_error": ev["std_error"]})
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    x = [r["epsilon"] for r in rows]
    axes[0].plot(x, [r["greedy_freq"] for r in rows], "o-", label="thực nghiệm")
    axes[0].plot(x, [r["theory"] for r in rows], "k--", label="lý thuyết 1-ε+ε/2")
    axes[0].set_xlabel("epsilon"); axes[0].set_ylabel("tần suất chọn greedy action")
    axes[0].set_title("Epsilon-greedy: tần suất chọn greedy"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].errorbar(x, [r["mean_reward"] for r in rows], yerr=[1.96 * r["std_error"] for r in rows],
                     fmt="o-", capsize=3)
    axes[1].set_xlabel("epsilon (lúc train)"); axes[1].set_ylabel("mean reward của policy học được")
    axes[1].set_title(f"MC control ({n_train:,} episode) theo epsilon"); axes[1].grid(alpha=0.3)
    fig.tight_layout()
    save_fig(fig, "epsilon_comparison.png")
    return rows


def plot_policy_performance(results, name="policy_performance.png"):
    """
    Bai 34: ve so sanh cac policy. `results` = {ten_policy: dict tu evaluate_policy}.
    Trai: mean reward (+- 95% CI). Phai: ti le win/draw/loss.
    """
    names = list(results)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    means = [results[n]["mean_reward"] for n in names]
    errs = [1.96 * results[n]["std_error"] for n in names]
    axes[0].bar(names, means, yerr=errs, capsize=4, color="tab:blue")
    axes[0].axhline(0, color="k", lw=0.8)
    axes[0].set_ylabel("mean reward"); axes[0].set_title("Mean reward (±95% CI)")
    axes[0].tick_params(axis="x", rotation=20)
    bottom = np.zeros(len(names))
    for key, color in (("win_rate", "tab:green"), ("draw_rate", "tab:gray"), ("loss_rate", "tab:red")):
        vals = np.array([results[n][key] for n in names])
        axes[1].bar(names, vals, bottom=bottom, label=key.replace("_rate", ""), color=color)
        bottom += vals
    axes[1].set_title("Tỉ lệ win / draw / loss"); axes[1].legend()
    axes[1].tick_params(axis="x", rotation=20)
    fig.tight_layout()
    save_fig(fig, name)

## PHẦN A — Làm quen với episodic environment

### Bài 1. Tạo Blackjack-v1

Tạo environment, reset và in observation, info, observation_space, action_space.

In [ ]:
env = make_env("Blackjack-v1")
observation, info = env.reset(seed=42)
print("observation      :", observation)
print("info             :", info)
print("observation_space:", env.observation_space)
print("action_space     :", env.action_space)
env.close()

### Bài 2. Phân tích observation

Observation là tuple `(player_sum, dealer_card, usable_ace)`. In 12 observation từ 12 episode khác nhau.

In [ ]:
env = make_env("Blackjack-v1")
# observation = (player_sum, dealer_card, usable_ace)
#   player_sum  : tổng điểm bài của người chơi (Ace tính 11 nếu không làm quá 21)
#   dealer_card : lá bài dealer đang lật ngửa (1 = Ace, 2..10; J/Q/K tính 10)
#   usable_ace  : True nếu người chơi có Ace đang được tính là 11 mà chưa bị bust
for episode_id in range(12):
    obs, _ = env.reset(seed=episode_id)      # mỗi episode = một ván bài mới
    print(f"episode {episode_id:2d}: player_sum={obs[0]:2d}, dealer_card={obs[1]:2d}, usable_ace={obs[2]}")
env.close()

### Bài 3. Phân tích action

In `action_space` và xác định ý nghĩa từng action.

In [ ]:
env = make_env("Blackjack-v1")
print("action_space:", env.action_space, "-> số action =", env.action_space.n)
# 0 = stick: dừng rút bài, dealer chơi nốt và ván kết thúc
# 1 = hit  : rút thêm một lá bài, ván tiếp tục (tổng > 21 thì bust và thua)
ACTION_NAMES = {0: "stick", 1: "hit"}
for a, name in ACTION_NAMES.items():
    print(f"action {a} = {name}")
env.close()

### Bài 4. Một episode ngẫu nhiên

Chạy một episode bằng random policy, in từng bước.

In [ ]:
env = make_env("Blackjack-v1")
rng = np.random.default_rng(0)
state, _ = env.reset(seed=0)
step = 0
while True:
    action = int(rng.integers(env.action_space.n))            # random policy
    next_state, reward, terminated, truncated, _ = env.step(action)
    print(f"step {step}: state={state} action={ACTION_NAMES[action]} reward={reward} "
          f"next_state={next_state} terminated={terminated} truncated={truncated}")
    state, step = next_state, step + 1
    if terminated or truncated:
        break
env.close()

### Bài 5. Lưu episode

Hàm `generate_episode(env, policy, seed)` (nằm trong `mc_utils.py`) trả về list `(state, action, reward)`.

In [ ]:
env = make_env("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, np.random.default_rng(1))
episode = sample_long_episode(env, random_policy, min_len=3)   # tìm episode đủ dài để minh họa
print("Số bước trong episode:", len(episode))
for t, (state, action, reward) in enumerate(episode):
    print(f"t={t}: state={state}, action={ACTION_NAMES[action]}, reward={reward}")
env.close()

## PHẦN B — Return từ episode

### Bài 6. Chuỗi reward

Lấy `rewards` từ episode; in độ dài episode, rewards và total reward.

In [ ]:
env = make_env("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, np.random.default_rng(1))
episode = sample_long_episode(env, random_policy, min_len=3)
rewards = [reward for _, _, reward in episode]
print("Độ dài episode:", len(episode))
print("rewards       :", rewards)
print("Tổng reward   :", sum(rewards))
env.close()

### Bài 7. Tính return từ cuối episode

`compute_returns(rewards, gamma)` duyệt ngược: `G_t = R_(t+1) + gamma * G_(t+1)`.

In [ ]:
env = make_env("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, np.random.default_rng(1))
episode = sample_long_episode(env, random_policy, min_len=3)
rewards = [reward for _, _, reward in episode]
for gamma in (1.0, 0.9):
    print(f"gamma={gamma}: rewards={rewards} -> G = {compute_returns(rewards, gamma)}")
env.close()

### Bài 8. Kiểm tra return

Kiểm tra tay với `rewards = [0, 0, 1]`: G_2 = 1, G_1 = gamma, G_0 = gamma^2.

In [ ]:
rewards = [0, 0, 1]
for gamma in (1.0, 0.9, 0.5):
    G = compute_returns(rewards, gamma)
    expected = [gamma ** 2, gamma, 1.0]
    assert np.allclose(G, expected), (G, expected)
    print(f"gamma={gamma}: G = {G}   (kỳ vọng {expected})")
print("Tất cả kiểm tra đều đúng.")

### Bài 9. Gắn return vào trajectory

Chuyển `(state, action, reward)` thành `(state, action, reward, G_t)`.

In [ ]:
env = make_env("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, np.random.default_rng(1))
episode = sample_long_episode(env, random_policy, min_len=3)
returns = compute_returns([r for _, _, r in episode], gamma=1.0)
trajectory = [(s, a, r, g) for (s, a, r), g in zip(episode, returns)]
for t, (s, a, r, g) in enumerate(trajectory):
    print(f"t={t}: state={s}, action={ACTION_NAMES[a]}, reward={r}, G_t={g}")
env.close()

### Bài 10. So sánh gamma

Cùng random policy, nhiều episode; tính mean `G_0` theo từng gamma và vẽ biểu đồ.

In [ ]:
env = make_env("Blackjack-v1")
rng = np.random.default_rng(0)
random_policy = make_random_policy(env.action_space.n, rng)
all_rewards = []
for i in range(5000):
    ep = generate_episode(env, random_policy, seed=0 if i == 0 else None)
    all_rewards.append([r for _, _, r in ep])

gammas = [0.5, 0.8, 0.9, 0.99, 1.0]
mean_g0 = []
for gamma in gammas:
    g0 = [compute_returns(rw, gamma)[0] for rw in all_rewards]
    mean_g0.append(np.mean(g0))
    print(f"gamma={gamma:4}: mean G_0 = {mean_g0[-1]:.4f}")

# Reward chỉ xuất hiện ở bước cuối nên G_0 = gamma^(T-1) * R_T. Episode Blackjack rất ngắn
# (1-3 bước) nên gamma ảnh hưởng ít; gamma nhỏ chỉ làm return "co" về 0.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(gammas, mean_g0, "o-")
ax.set_xlabel("gamma"); ax.set_ylabel("mean G_0"); ax.set_title("Mean G_0 theo gamma (random policy)")
ax.grid(alpha=0.3)
save_fig(fig, "gamma_comparison.png")
env.close()

## PHẦN C — Monte Carlo Prediction cơ bản

### Bài 11. Policy cố định

Kiểm tra ý nghĩa action trước khi dùng policy `stick_on_20_policy`: action 0 = stick, 1 = hit.

In [ ]:
print("Ý nghĩa action:", ACTION_NAMES)
for state in [(21, 5, False), (20, 10, False), (19, 10, False), (12, 6, False), (15, 3, True)]:
    action = stick_on_20_policy(state)
    print(f"state={state} -> action {action} ({ACTION_NAMES[action]})")

### Bài 12. Sinh 100 episode

Dùng `stick_on_20_policy`, đếm win/loss/draw.

In [ ]:
env = make_env("Blackjack-v1")
counts = {"win": 0, "loss": 0, "draw": 0}
for i in range(100):
    ep = generate_episode(env, stick_on_20_policy, seed=0 if i == 0 else None)
    final_reward = ep[-1][2]
    if final_reward > 0:   counts["win"] += 1
    elif final_reward < 0: counts["loss"] += 1
    else:                  counts["draw"] += 1
for k, v in counts.items():
    print(f"{k:5s}: {v:3d}  ({v / 100:.0%})")
env.close()

### Bài 13. Thu thập return theo state

Với mỗi state trong episode, lưu return tương ứng vào `returns[state]` (chưa phân biệt first/every visit).

In [ ]:
env = make_env("Blackjack-v1")
returns = defaultdict(list)
for i in range(10000):
    ep = generate_episode(env, stick_on_20_policy, seed=0 if i == 0 else None)
    G = compute_returns([r for _, _, r in ep], gamma=1.0)
    for (state, _, _), g in zip(ep, G):
        returns[state].append(g)
print("Số state đã gặp:", len(returns))
example = (20, 10, False)
print(f"returns[{example}]: {len(returns[example])} mẫu, 10 mẫu đầu = {returns[example][:10]}")
env.close()

### Bài 14. Ước lượng V(s)

`V[state] = np.mean(returns[state])`; in 10 state xuất hiện nhiều nhất.

In [ ]:
env = make_env("Blackjack-v1")
returns = defaultdict(list)
for i in range(10000):
    ep = generate_episode(env, stick_on_20_policy, seed=0 if i == 0 else None)
    G = compute_returns([r for _, _, r in ep], gamma=1.0)
    for (state, _, _), g in zip(ep, G):
        returns[state].append(g)
V = {state: np.mean(g_list) for state, g_list in returns.items()}
top10 = sorted(returns, key=lambda s: len(returns[s]), reverse=True)[:10]
for s in top10:
    print(f"V{s} = {V[s]:+.3f}   (từ {len(returns[s])} return)")
env.close()

### Bài 15. Theo dõi một state cụ thể

Theo dõi estimate V(20, 10, False) sau 100/500/1000/5000/10000 episode và vẽ đường hội tụ.

In [ ]:
env = make_env("Blackjack-v1")
target_state = (20, 10, False)
checkpoints = [100, 500, 1000, 5000, 10000]
total, count, estimates = 0.0, 0, []
for i in range(1, 10001):
    ep = generate_episode(env, stick_on_20_policy, seed=0 if i == 1 else None)
    G = compute_returns([r for _, _, r in ep], gamma=1.0)
    for (state, _, _), g in zip(ep, G):
        if state == target_state:
            total += g
            count += 1
    if i in checkpoints:
        estimates.append(total / count if count else np.nan)
        print(f"sau {i:5d} episode: V{target_state} ≈ {estimates[-1]:+.4f}  (n={count})")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(checkpoints, estimates, "o-")
ax.set_xscale("log"); ax.set_xlabel("số episode"); ax.set_ylabel("V(s) ước lượng")
ax.set_title(f"Hội tụ của V{target_state}"); ax.grid(alpha=0.3)
save_fig(fig, "convergence_target_state.png")
env.close()

## PHẦN D — First-Visit Monte Carlo

### Bài 16. Phát hiện first visit

Xác định vị trí xuất hiện đầu tiên của mỗi state trong một episode (tự viết, không dùng thư viện RL).

In [ ]:
def find_first_visits(episode):
    """Trả về dict {state: chỉ số timestep xuất hiện ĐẦU TIÊN}."""
    first_visit_index = {}
    for t, (state, _, _) in enumerate(episode):
        if state not in first_visit_index:
            first_visit_index[state] = t
    return first_visit_index

env = make_env("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, np.random.default_rng(1))
episode = sample_long_episode(env, random_policy, min_len=3)
print("Blackjack:", find_first_visits(episode))
print("-> Trong Blackjack mỗi lần hit đều đổi (player_sum, usable_ace) và không bao giờ quay lại state cũ => state KHÔNG lặp lại trong 1 episode.")

# FrozenLake: agent có thể quay lại cùng một ô => state lặp lại, first visit khác every visit
frozen = make_env("FrozenLake-v1")
frozen_policy = make_random_policy(4, np.random.default_rng(0))
ep = sample_long_episode(frozen, frozen_policy, min_len=10)
states = [s for s, _, _ in ep]
print("\nFrozenLake states   :", states)
print("first-visit index   :", find_first_visits(ep))
repeated = sorted({s for s in states if states.count(s) > 1})
print("state xuất hiện >1 lần:", repeated)
env.close(); frozen.close()

### Bài 17. First-Visit MC Prediction

`first_visit_mc_prediction(env, policy, n_episodes, gamma)` trả về `V, returns_count`.

In [ ]:
env = make_env("Blackjack-v1")
V, returns_count = first_visit_mc_prediction(env, stick_on_20_policy, n_episodes=5000, gamma=1.0, seed=0)
print("Số state có estimate:", len(V))
print("Tổng số lần cập nhật (first visit):", sum(returns_count.values()))
for s in sorted(V, key=lambda s: -returns_count[s])[:8]:
    print(f"V{s} = {V[s]:+.3f}  (n={returns_count[s]})")
env.close()

### Bài 18. Chạy First-Visit MC

Chạy với 100, 1000, 10000, 50000 episode; in số state đã được ước lượng.

In [ ]:
env = make_env("Blackjack-v1")
for n in [100, 1000, 10000, 50000]:
    V, returns_count = first_visit_mc_prediction(env, stick_on_20_policy, n_episodes=n, seed=0)
    print(f"{n:6d} episode -> {len(V):3d} state được ước lượng, state ít mẫu nhất có n={min(returns_count.values())}")
env.close()

### Bài 19. Visualize value

Heatmap V(s) theo `player_sum` × `dealer_card` (usable_ace = False, và True để tham khảo).

In [ ]:
env = make_env("Blackjack-v1")
V, _ = first_visit_mc_prediction(env, stick_on_20_policy, n_episodes=100_000, seed=0)
plot_value_heatmap(V, False, "blackjack_value_no_usable_ace.png", "V(s), stick_on_20, không usable ace")
plot_value_heatmap(V, True, "blackjack_value_usable_ace.png", "V(s), stick_on_20, có usable ace")
# Bảng số cho usable_ace=False (hàng: player_sum 12..21, cột: dealer A,2..10)
grid = value_grid(V, usable_ace=False)
print(np.round(grid, 2))
env.close()

## PHẦN E — Every-Visit Monte Carlo

### Bài 20. Every-Visit MC Prediction

`every_visit_mc_prediction(...)`: dùng return ở MỌI lần state xuất hiện.

In [ ]:
env = make_env("Blackjack-v1")
V, returns_count = every_visit_mc_prediction(env, stick_on_20_policy, n_episodes=5000, gamma=1.0, seed=0)
print("Số state có estimate:", len(V))
for s in sorted(V, key=lambda s: -returns_count[s])[:8]:
    print(f"V{s} = {V[s]:+.3f}  (n={returns_count[s]})")
env.close()

### Bài 21. So sánh First-Visit và Every-Visit

Cùng policy, gamma, n_episodes và seed; so sánh V(s) tại 10 state.

In [ ]:
env = make_env("Blackjack-v1")
V_first, _ = first_visit_mc_prediction(env, stick_on_20_policy, 50_000, gamma=1.0, seed=0)
V_every, cnt = every_visit_mc_prediction(env, stick_on_20_policy, 50_000, gamma=1.0, seed=0)
print(f"{'state':18s} {'V_first':>9s} {'V_every':>9s} {'|diff|':>8s}")
for s in sorted(V_every, key=lambda s: -cnt[s])[:10]:
    print(f"{str(s):18s} {V_first[s]:9.4f} {V_every[s]:9.4f} {abs(V_first[s] - V_every[s]):8.5f}")
env.close()

### Bài 22. Sai khác giữa hai phương pháp

Mean absolute difference trên các state chung — trên Blackjack và trên FrozenLake.

In [ ]:
def mean_abs_diff(V_a, V_b):
    common = set(V_a) & set(V_b)
    return np.mean([abs(V_a[s] - V_b[s]) for s in common]), len(common)

env = make_env("Blackjack-v1")
Vf, _ = first_visit_mc_prediction(env, stick_on_20_policy, 50_000, seed=0)
Ve, _ = every_visit_mc_prediction(env, stick_on_20_policy, 50_000, seed=0)
d, n = mean_abs_diff(Vf, Ve)
print(f"Blackjack : mean |V_first - V_every| = {d:.6f} trên {n} state chung")
# = 0 vì state không lặp lại trong 1 episode => first visit và every visit chọn đúng cùng các return.

frozen = make_env("FrozenLake-v1")
pf = make_random_policy(4, np.random.default_rng(0)); Vf, _ = first_visit_mc_prediction(frozen, pf, 20_000, seed=0)
pe = make_random_policy(4, np.random.default_rng(0)); Ve, _ = every_visit_mc_prediction(frozen, pe, 20_000, seed=0)
d, n = mean_abs_diff(Vf, Ve)
print(f"FrozenLake: mean |V_first - V_every| = {d:.6f} trên {n} state chung")
# > 0 vì agent có thể quay lại cùng một ô trong một episode.
env.close(); frozen.close()

### Bài 23. Biểu đồ hội tụ

Ba state mỗi môi trường, V(s) của First-Visit và Every-Visit theo số episode. Lưu `first_vs_every_visit.png`.

In [ ]:
blackjack = make_env("Blackjack-v1")
frozen = make_env("FrozenLake-v1")
diffs = compare_first_every_convergence(blackjack, frozen, n_episodes=10_000, seed=0)
print("Mean absolute difference:", diffs)
blackjack.close(); frozen.close()

## PHẦN F — Action-value function Q(s,a)

### Bài 24. Lưu state-action-return

Lưu `(state, action) -> list return` bằng `defaultdict(list)` (dùng random policy để có cả hai action).

In [ ]:
env = make_env("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, np.random.default_rng(1))
sa_returns = defaultdict(list)
for i in range(20000):
    ep = generate_episode(env, random_policy, seed=1 if i == 0 else None)
    G = compute_returns([r for _, _, r in ep], gamma=1.0)
    seen = set()
    for t, (state, action, _) in enumerate(ep):
        if (state, action) not in seen:          # first visit trên cặp (s, a)
            seen.add((state, action))
            sa_returns[(state, action)].append(G[t])
print("Số cặp (state, action):", len(sa_returns))
for key in [((20, 10, False), 0), ((20, 10, False), 1), ((13, 2, False), 1)]:
    r = sa_returns[key]
    print(f"{key}: n={len(r)}, mean return = {np.mean(r):+.3f}")
env.close()

### Bài 25. Ước lượng Q(s,a)

`mc_action_value_prediction` ước lượng `Q[state][action]` cho random policy.

In [ ]:
env = make_env("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, np.random.default_rng(1))
Q, N = mc_action_value_prediction(env, random_policy, n_episodes=100_000, gamma=1.0, seed=1)
print("Số state có Q:", len(Q))
for s in [(20, 10, False), (13, 2, False), (18, 6, True), (12, 6, False)]:
    print(f"Q{s} = [stick={Q[s][0]:+.3f}, hit={Q[s][1]:+.3f}]   N = {N[s]}")
env.close()

### Bài 26. Greedy action từ Q

`greedy_action(Q, state, n_actions)` dùng `np.argmax`.

In [ ]:
env = make_env("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, np.random.default_rng(1))
Q, _ = mc_action_value_prediction(env, random_policy, n_episodes=100_000, seed=1)
for s in [(20, 10, False), (13, 2, False), (18, 6, True), (12, 6, False)]:
    a = greedy_action(Q, s, env.action_space.n)
    print(f"state={s}: Q={np.round(Q[s], 3)} -> greedy action = {a} ({ACTION_NAMES[a]})")
# Lưu ý: đây là greedy theo Q của RANDOM policy, chưa phải policy tối ưu.
env.close()

## PHẦN G — Epsilon-greedy và Exploration

### Bài 27. Epsilon-greedy policy

`epsilon_greedy_action(Q, state, n_actions, epsilon, rng)`: xác suất ε chọn ngẫu nhiên, còn lại chọn greedy.

In [ ]:
rng = np.random.default_rng(0)
Q = defaultdict(lambda: np.zeros(2))
Q[(18, 6, False)] = np.array([0.15, -0.20])      # stick tốt hơn => greedy action = 0
for eps in (0.0, 0.3, 1.0):
    actions = [epsilon_greedy_action(Q, (18, 6, False), 2, eps, rng) for _ in range(20)]
    print(f"epsilon={eps}: {actions}")

### Bài 28. Kiểm tra epsilon-greedy

Q giả `[1.0, 5.0]`, sinh 10000 action với ε = 0.1.

In [ ]:
rng = np.random.default_rng(0)
Q = {"s": np.array([1.0, 5.0])}
epsilon, n = 0.1, 10_000
actions = np.array([epsilon_greedy_action(Q, "s", 2, epsilon, rng) for _ in range(n)])
counts = np.bincount(actions, minlength=2)
for a in range(2):
    print(f"action {a}: {counts[a]:5d} lần  ({counts[a] / n:.3f})")
print(f"Lý thuyết: greedy = 1 - ε + ε/2 = {1 - epsilon + epsilon / 2:.3f}, còn lại = ε/2 = {epsilon / 2:.3f}")
# Nhận xét: action 1 (greedy) được chọn ~95%, action 0 ~5% => agent vẫn thỉnh thoảng khám phá.

### Bài 29. So sánh epsilon

Tần suất chọn greedy theo ε và chất lượng policy học được. Lưu `epsilon_comparison.png`.

In [ ]:
env = make_env("Blackjack-v1")
rows = epsilon_experiment(env, epsilons=(0.01, 0.05, 0.10, 0.20, 0.50), n_train=100_000, n_eval=20_000)
print(f"{'epsilon':>8s} {'greedy_freq':>12s} {'lý thuyết':>10s} {'mean reward (learned)':>22s}")
for r in rows:
    print(f"{r['epsilon']:8.2f} {r['greedy_freq']:12.3f} {r['theory']:10.3f} {r['mean_reward']:14.4f} ± {1.96 * r['std_error']:.4f}")
env.close()

## PHẦN H — On-policy Monte Carlo Control

### Bài 30. Khởi tạo Q

Khởi tạo `Q`, `N` (đếm số lần cập nhật) để dùng incremental mean.

In [ ]:
env = make_env("Blackjack-v1")
Q = defaultdict(lambda: np.zeros(env.action_space.n))
N = defaultdict(lambda: np.zeros(env.action_space.n))      # N(s,a)
print("Q[(18, 6, False)] khi chưa học :", Q[(18, 6, False)])
print("N[(18, 6, False)] khi chưa học :", N[(18, 6, False)])
print("Số state hiện có trong Q       :", len(Q))
env.close()

### Bài 31. Incremental mean

`update_q_incremental`: `N += 1; Q += (G - Q) / N`. Kiểm tra bằng trung bình thường.

In [ ]:
Q = defaultdict(lambda: np.zeros(2))
N = defaultdict(lambda: np.zeros(2))
state, action = (18, 6, False), 1
returns_seen = []
for G in [1, -1, 1, 1, -1, -1, 1]:
    returns_seen.append(G)
    update_q_incremental(Q, N, state, action, G)
    assert np.isclose(Q[state][action], np.mean(returns_seen))
    print(f"G={G:+d}: Q={Q[state][action]:+.4f}  mean(returns)={np.mean(returns_seen):+.4f}  N={int(N[state][action])}")
print("Incremental mean trùng với trung bình đầy đủ, nhưng không cần lưu toàn bộ returns.")

### Bài 32. On-policy First-Visit MC Control

`on_policy_mc_control(env, n_episodes, gamma, epsilon, seed)` trả về `Q, policy, episode_rewards`. Demo nhanh với 20000 episode.

In [ ]:
env = make_env("Blackjack-v1")
Q, policy, episode_rewards = on_policy_mc_control(env, n_episodes=20_000, gamma=1.0, epsilon=0.1, seed=42)
print("Số state đã học   :", len(policy))
print("Mean reward 1000 episode cuối (lúc train, có explore):", np.mean(episode_rewards[-1000:]))
env.close()

### Bài 33. Học policy cho Blackjack

Train 300000 episode (≥ 100000), in policy tại các state ví dụ. Policy lấy từ `Q`, không hard-code.

In [ ]:
env = make_env("Blackjack-v1")
Q, policy, episode_rewards = load_or_train_control(env, n_episodes=300_000, gamma=1.0, epsilon=0.1, seed=42)
print("Số state đã học:", len(policy))
for s in [(20, 10, False), (18, 6, False), (13, 2, False), (18, 6, True)]:
    if s in policy:
        print(f"state={s}: Q={np.round(Q[s], 3)} -> {ACTION_NAMES[policy[s]]}")
    else:
        print(f"state={s}: chưa gặp trong lúc train")
env.close()

### Bài 34. Đánh giá policy đã học

`evaluate_policy` trên episode mới: so sánh random / fixed / MC learned. Lưu `policy_performance.png`.

In [ ]:
env = make_env("Blackjack-v1")
Q, policy, _ = load_or_train_control(env, n_episodes=300_000, epsilon=0.1, seed=42)
n_eval = 50_000
results = {
    "random": evaluate_policy(env, make_random_policy(2, np.random.default_rng(0)), n_eval, seed=123),
    "fixed (stick>=20)": evaluate_policy(env, stick_on_20_policy, n_eval, seed=123),
    "fixed (stick>=17)": evaluate_policy(env, stick_on_17_policy, n_eval, seed=123),
    "MC learned": evaluate_policy(env, make_policy_fn(policy), n_eval, seed=123),
}
print(f"{'policy':20s} {'win':>7s} {'loss':>7s} {'draw':>7s} {'mean reward':>12s}")
for name, r in results.items():
    print(f"{name:20s} {r['win_rate']:7.3f} {r['loss_rate']:7.3f} {r['draw_rate']:7.3f} {r['mean_reward']:12.4f}")
plot_policy_performance(results)
env.close()

### Bài 35. Learning curve

Moving average (window = 1000) của `episode_rewards` lúc training. Lưu `mc_convergence.png`.

In [ ]:
env = make_env("Blackjack-v1")
Q, policy, episode_rewards = load_or_train_control(env, n_episodes=300_000, epsilon=0.1, seed=42)
window = 1000
ma = moving_average(episode_rewards, window)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(len(ma)) + window, ma, label="moving average (window=1000)")
ax.axhline(np.mean(episode_rewards[:window]), color="r", ls=":", label="1000 episode đầu")
ax.set_xlabel("episode"); ax.set_ylabel("mean reward")
ax.set_title("On-policy MC control: learning curve (epsilon=0.1)")
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, "mc_convergence.png")
print(f"1000 episode đầu : {np.mean(episode_rewards[:1000]):+.3f}")
print(f"1000 episode cuối: {np.mean(episode_rewards[-1000:]):+.3f}   (lúc train vẫn còn explore ε=0.1)")
env.close()

## PHẦN I — Mini-project

### Bài 36. Monte Carlo Agent hoàn chỉnh

Mini-project 20 bước (xem `main.py`). Gồm cả môi trường phụ FrozenLake-v1.

In [ ]:
def run_mini_project(n_pred=50_000, n_train=300_000, n_eval=50_000, seed=42):
    """Bai 36: Monte Carlo agent hoan chinh (20 buoc) tren Blackjack-v1 (+ FrozenLake-v1 lam moi truong phu)."""
    print("=== 1. Environment ===")
    env = make_env("Blackjack-v1")
    print("Môi trường chính: Blackjack-v1")

    print("\n=== 2. Observation / action space ===")
    print("observation_space:", env.observation_space)
    print("action_space     :", env.action_space, "->", ACTION_NAMES)

    print("\n=== 3-4. Episode mẫu & trajectory ===")
    random_policy = make_random_policy(env.action_space.n, np.random.default_rng(seed))
    episode = sample_long_episode(env, random_policy, min_len=2)
    for t, (s, a, r) in enumerate(episode):
        print(f"  t={t}: state={s}, action={ACTION_NAMES[a]}, reward={r}")

    print("\n=== 5-7. Reward, return, discounted return ===")
    rewards = [r for _, _, r in episode]
    print("rewards:", rewards)
    for gamma in (1.0, 0.9):
        print(f"  gamma={gamma}: G_t =", [round(g, 3) for g in compute_returns(rewards, gamma)])

    print("\n=== 8. Fixed policy (stick khi tổng >= 20) ===")
    fixed = evaluate_policy(env, stick_on_20_policy, 10_000, seed=1)
    print("  ", {k: round(v, 4) for k, v in fixed.items()})

    print("\n=== 9-11. First-Visit vs Every-Visit MC prediction ===")
    V_f, cnt_f = first_visit_mc_prediction(env, stick_on_20_policy, n_pred, seed=seed)
    V_e, cnt_e = every_visit_mc_prediction(env, stick_on_20_policy, n_pred, seed=seed)
    common = set(V_f) & set(V_e)
    mad = np.mean([abs(V_f[s] - V_e[s]) for s in common])
    print(f"  số state ước lượng: first={len(V_f)}, every={len(V_e)}; mean |V_first - V_every| = {mad:.6f}")
    print("  (Blackjack: state không lặp lại trong 1 episode => hai phương pháp trùng nhau)")
    plot_value_heatmap(V_f, False, "blackjack_value_no_usable_ace.png", "V(s) stick_on_20, không có usable ace")
    plot_value_heatmap(V_f, True, "blackjack_value_usable_ace.png", "V(s) stick_on_20, có usable ace")

    print("\n=== 12. Q(s,a) của random policy ===")
    Q_rand, N_rand = mc_action_value_prediction(env, random_policy, 100_000, seed=seed)
    for s in [(20, 10, False), (13, 2, False), (18, 6, True)]:
        print(f"  Q{s} = {np.round(Q_rand[s], 3)}  (stick, hit)")

    print("\n=== 13. Epsilon-greedy ===")
    rng = np.random.default_rng(seed)
    acts = [epsilon_greedy_action({"s": np.array([1.0, 5.0])}, "s", 2, 0.1, rng) for _ in range(10_000)]
    print(f"  epsilon=0.1, Q=[1,5]: tần suất greedy = {np.mean(np.array(acts) == 1):.3f} (lý thuyết 0.95)")

    print("\n=== 14-15. On-policy MC control + learning curve ===")
    Q, policy, ep_rewards = load_or_train_control(env, n_train, epsilon=0.1, seed=seed)
    print(f"  đã train {n_train:,} episode, học được policy cho {len(policy)} state")
    ma = moving_average(ep_rewards, 1000)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.arange(len(ma)) + 1000, ma)
    ax.set_xlabel("episode"); ax.set_ylabel("mean reward (moving avg, window=1000)")
    ax.set_title("On-policy MC control: learning curve (epsilon=0.1)"); ax.grid(alpha=0.3)
    save_fig(fig, "mc_convergence.png")

    print("\n=== 16. Learned policy ===")
    learned_fn = make_policy_fn(policy)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    plot_policy_heatmap(policy, False, axes[0], "Learned policy, không usable ace (S=stick, H=hit)")
    plot_policy_heatmap(policy, True, axes[1], "Learned policy, có usable ace")
    fig.tight_layout()
    save_fig(fig, "blackjack_policy.png")

    print("\n=== 17-18. Evaluation & so sánh policy ===")
    results = {
        "random": evaluate_policy(env, make_random_policy(2, np.random.default_rng(seed)), n_eval, seed=123),
        "fixed (stick>=20)": evaluate_policy(env, stick_on_20_policy, n_eval, seed=123),
        "fixed (stick>=17)": evaluate_policy(env, stick_on_17_policy, n_eval, seed=123),
        "MC learned": evaluate_policy(env, learned_fn, n_eval, seed=123),
    }
    print(f"  {'policy':20s} {'win':>7s} {'loss':>7s} {'draw':>7s} {'mean reward':>12s}")
    for name, r in results.items():
        print(f"  {name:20s} {r['win_rate']:7.3f} {r['loss_rate']:7.3f} {r['draw_rate']:7.3f} {r['mean_reward']:12.4f}")
    plot_policy_performance(results)

    print("\n=== Môi trường phụ: FrozenLake-v1 (generate episode, return, MC prediction) ===")
    frozen = make_env("FrozenLake-v1")
    frozen_policy = make_random_policy(frozen.action_space.n, np.random.default_rng(seed))
    ep = generate_episode(frozen, frozen_policy, seed=seed)
    print(f"  episode mẫu dài {len(ep)} bước, return G_0 = {compute_returns([r for _, _, r in ep], 0.99)[0]:.3f}")
    Vf_f, _ = first_visit_mc_prediction(frozen, make_random_policy(4, np.random.default_rng(seed)), 20_000, seed=seed)
    Ve_f, _ = every_visit_mc_prediction(frozen, make_random_policy(4, np.random.default_rng(seed)), 20_000, seed=seed)
    mad_f = np.mean([abs(Vf_f[s] - Ve_f[s]) for s in set(Vf_f) & set(Ve_f)])
    print(f"  V(0) first={Vf_f[0]:.4f}, every={Ve_f[0]:.4f}; mean |diff| = {mad_f:.5f} (FrozenLake có state lặp => khác nhau)")

    print("\n=== 19. Nhận xét ===")
    best = results["MC learned"]["mean_reward"]
    print(f"  - MC learned (mean reward {best:.3f}) vượt xa random ({results['random']['mean_reward']:.3f})"
          f" và fixed stick>=20 ({results['fixed (stick>=20)']['mean_reward']:.3f}).")
    print("  - Blackjack có lợi thế nhà cái nên mean reward tối ưu vẫn hơi âm (khoảng -0.05).")
    print("  - Learned policy có nhiễu ở các state hiếm; cần nhiều episode hơn để ổn định.")

    print("\n=== 20. Kết luận ===")
    print("  Chỉ từ các episode (không dùng model), Monte Carlo control với epsilon-greedy học được policy")
    print("  gần tối ưu cho Blackjack. Hạn chế: phải chờ hết episode và phương sai return lớn => Lab04: TD.")
    env.close(); frozen.close()
    return results

# Bài 36: chạy mini-project
results = run_mini_project()

## Câu hỏi bắt buộc

**1. Monte Carlo method trong RL là gì?**

Là nhóm phương pháp ước lượng value function và tìm policy từ **các episode đã quan sát**, bằng cách lấy **trung bình return** thực tế thu được. Không cần biết mô hình môi trường.

**2. Vì sao Monte Carlo được gọi là model-free?**

Vì nó không cần transition probability `P(s'|s,a)` hay reward model; chỉ cần mẫu `(S, A, R)` do agent tương tác tạo ra. Trong bài, `generate_episode` chỉ gọi `env.step()`.

**3. Vì sao Monte Carlo cần episodic task?**

Return `G_t` là tổng reward đến cuối episode nên chỉ tính được khi episode **kết thúc**. Task không kết thúc (hoặc quá dài) phải giới hạn bằng `max_steps`.

**4. Episode khác trajectory như thế nào trong bài này?**

Episode là toàn bộ một lượt tương tác từ trạng thái đầu đến trạng thái kết thúc. Trajectory là chuỗi `(state, action, reward)` ghi lại diễn biến đó; ở đây trajectory của một episode được lưu thành list, sau đó gắn thêm `G_t`.

**5. Return được tính khi nào?**

Sau khi episode kết thúc, duyệt **ngược** từ cuối: `G_t = R_(t+1) + γ·G_(t+1)`.

**6. Monte Carlo Prediction dùng để làm gì?**

Đánh giá một policy cố định `π`: ước lượng `V_π(s)` (hoặc `Q_π(s,a)`) bằng trung bình các return quan sát được.

**7. First-Visit MC khác Every-Visit MC thế nào?**

First-Visit chỉ dùng return tại **lần đầu** state xuất hiện trong episode; Every-Visit dùng return ở **mọi lần** xuất hiện. Cả hai đều hội tụ về `V_π`. Trong Blackjack state không lặp lại trong một episode nên kết quả trùng nhau (Bài 22); ở FrozenLake có state lặp nên khác nhau nhẹ.

**8. Vì sao phải lấy trung bình nhiều return?**

Mỗi return là một mẫu ngẫu nhiên (do bài rút ngẫu nhiên). `V(s)` là **kỳ vọng** của return, nên cần trung bình nhiều mẫu để giảm phương sai (luật số lớn).

**9. Khi số episode tăng, estimate V(s) thay đổi như thế nào?**

Dao động mạnh lúc đầu rồi ổn định dần về giá trị thật (sai số ~ 1/√n) — xem biểu đồ Bài 15 và Bài 23.

**10. V(s) khác Q(s,a) thế nào?**

`V(s)` là giá trị kỳ vọng khi ở state `s` và theo policy; `Q(s,a)` là giá trị kỳ vọng khi ở `s`, làm action `a` trước rồi mới theo policy.

**11. Vì sao MC control cần Q(s,a) thay vì chỉ V(s)?**

Để cải thiện policy chỉ bằng mẫu mà không có model: muốn biết action nào tốt hơn phải so sánh `Q(s,a)` giữa các action. Với `V(s)` cần model `P` để nhìn trước một bước.

**12. Exploration là gì?**

Thử các action không phải tốt nhất hiện tại để thu thêm thông tin, tránh kẹt ở ước lượng sai.

**13. Exploitation là gì?**

Chọn action có Q cao nhất hiện biết để tối đa hóa reward.

**14. Epsilon-greedy cân bằng exploration/exploitation thế nào?**

Với xác suất ε chọn action ngẫu nhiên (explore), với 1−ε chọn greedy (exploit). Mọi action luôn có xác suất ≥ ε/|A| được chọn. Bài 28: ε = 0.1, 2 action → greedy ≈ 0.95.

**15. Khi epsilon quá lớn thì có vấn đề gì?**

Agent hành động ngẫu nhiên quá nhiều: reward lúc train thấp, Q học theo policy rất nhiễu (với ε = 0.5 thì Q là của policy ε-soft chứ không phải greedy).

**16. Khi epsilon quá nhỏ từ đầu thì có vấn đề gì?**

Ít khám phá: các cặp (s,a) ít được thử, Q sai có thể không được sửa; hội tụ chậm hoặc kẹt ở policy kém. Ở Blackjack, ε = 0.01 cho kết quả kém hơn các ε vừa (Bài 29).

**17. Incremental mean có lợi gì so với lưu toàn bộ returns?**

Bộ nhớ O(1) cho mỗi (s,a) thay vì O(số return); cập nhật `Q += (G − Q)/N` cho kết quả đúng bằng trung bình đầy đủ (Bài 31) và dễ mở rộng sang step-size cố định.

**18. On-policy MC control nghĩa là gì?**

Policy dùng để sinh episode (epsilon-greedy theo Q) cũng chính là policy đang được đánh giá và cải thiện.

**19. Policy dùng để sinh episode có thay đổi trong quá trình học không?**

Có. Mỗi lần Q cập nhật, epsilon-greedy theo Q thay đổi, nên các episode sau được sinh bởi policy tốt hơn.

**20. Monte Carlo có bootstrap không?**

Không. Nó dùng return thực tế đến cuối episode, không dùng ước lượng của state kế tiếp để cập nhật (khác TD/DP).

**21. MC phải đợi đến cuối episode vì sao?**

Vì cần đủ chuỗi reward để tính `G_t`. Chỉ biết return đầy đủ khi episode kết thúc.

**22. Random policy và learned policy khác nhau ra sao về mean reward?**

Learned policy cao hơn rõ rệt: random khoảng −0.40, learned khoảng −0.06 (xem bảng Bài 34). Vẫn âm vì Blackjack có lợi thế nhà cái.

**23. Learned policy có chắc chắn tối ưu không? Vì sao?**

Không chắc. Lý thuyết cần mọi (s,a) được thăm vô hạn lần và ε giảm dần (GLIE); ở đây dùng ε cố định, số episode hữu hạn nên state hiếm vẫn nhiễu, và policy greedy là tối ưu trong lớp ε-soft theo Q ước lượng.

**24. Số episode ảnh hưởng chất lượng estimate thế nào?**

Nhiều episode → mỗi state có nhiều mẫu hơn → sai số giảm, phủ nhiều state hơn (Bài 18). Ít episode (vài chục) không đủ để kết luận hội tụ.

**25. Monte Carlo khác Dynamic Programming ở điểm quan trọng nào?**

DP cần đầy đủ model và bootstrap (cập nhật dựa trên ước lượng kế tiếp); MC không cần model, học từ trải nghiệm và dùng return thực đến cuối episode, nhưng chỉ dùng được cho episodic task và có phương sai cao hơn.

## Kết luận

- Monte Carlo học trực tiếp từ episode, không cần model; First-Visit và Every-Visit cùng ước lượng `V_π`.
- On-policy MC control với epsilon-greedy học được policy cho Blackjack tốt hơn hẳn random và fixed policy `stick>=20`.
- Hạn chế: phải đợi hết episode, phương sai cao, cần đủ exploration → **Lab04: Temporal-Difference Learning**.